In [21]:
import numpy as np
import pandas as pd

In [22]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

cols = ["engine_displacement", "horsepower", "vehicle_weight", "model_year", "fuel_efficiency_mpg"]
df = df[cols]

In [23]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [24]:
df["horsepower"].median()

np.float64(254.0)

In [25]:
def prepare_X(df_, fillval):
    df_ = df_.copy()
    df_["horsepower"] = df_["horsepower"].fillna(fillval)
    return df_[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T @ X
    XTX = XTX + r * np.eye(XTX.shape[0])
    w = np.linalg.inv(XTX) @ X.T @ y
    return w[0], w[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

def split(df_, seed):
    n = len(df_)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    idx = np.arange(n)
    np.random.seed(seed)
    np.random.shuffle(idx)
    df_shuf = df_.iloc[idx].reset_index(drop=True)
    df_train = df_shuf.iloc[:n_train]
    df_val = df_shuf.iloc[n_train:n_train + n_val]
    df_test = df_shuf.iloc[n_train + n_val:]
    return df_train, df_val, df_test

In [26]:
df_train, df_val, df_test = split(df, 42)
y_train = df_train["fuel_efficiency_mpg"].values
y_val = df_val["fuel_efficiency_mpg"].values

for fillmethod in ["zero", "mean"]:
    fv = 0 if fillmethod == "zero" else df_train["horsepower"].mean()
    X_train = prepare_X(df_train, fv)
    X_val = prepare_X(df_val, fv)
    w0, w = train_linear_regression_reg(X_train, y_train, r=0)
    y_pred = w0 + X_val @ w
    print(fillmethod, round(rmse(y_val, y_pred), 2))

zero 2.21
mean 2.2


In [27]:
X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val @ w
    print(r, round(rmse(y_val, y_pred), 2))

0 2.21
0.01 2.21
0.1 2.22
1 2.35
5 2.41
10 2.42
100 2.43


In [28]:
scores = []
for seed in range(10):
    dtr, dv, dte = split(df, seed)
    ytr = dtr["fuel_efficiency_mpg"].values
    yv = dv["fuel_efficiency_mpg"].values
    Xtr = prepare_X(dtr, 0)
    Xv = prepare_X(dv, 0)
    w0, w = train_linear_regression_reg(Xtr, ytr, r=0)
    yp = w0 + Xv @ w
    scores.append(rmse(yv, yp))

round(np.std(scores), 3)

np.float64(0.029)

In [29]:
dtr, dv, dte = split(df, 9)
dfull = pd.concat([dtr, dv])
yfull = dfull["fuel_efficiency_mpg"].values
ytest = dte["fuel_efficiency_mpg"].values

Xfull = prepare_X(dfull, 0)
Xtest = prepare_X(dte, 0)

w0, w = train_linear_regression_reg(Xfull, yfull, r=0.001)
ypred = w0 + Xtest @ w
round(rmse(ytest, ypred), 3)

np.float64(2.236)